In [ ]:
%idle_timeout 20
%worker_type Standard
%number_of_workers 2

In [ ]:
# Load libraries
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import FloatType, IntegerType, StringType, NumericType
from pyspark.ml.feature import StringIndexer, OneHotEncoder, Tokenizer, HashingTF, IDF, VectorAssembler
from pyspark.sql.functions import col, log1p, percent_rank, length, sum
from pyspark.sql.window import Window
import pyspark.sql.functions as F

In [ ]:
spark = SparkSession.builder.appName("Amazon Product Data Cleaning").getOrCreate()

In [ ]:
bucket_name = 'amazon-product-dataset-2024'
categories_file_path = 's3://{}/amazon_categories.csv'.format(bucket_name)
products_cleaned__file_path = "s3://amazon-product-dataset-2024/cleaned/"

In [ ]:
categories = spark.read.csv(categories_file_path, header=True, inferSchema=True)
categories.show(5)

In [ ]:
df = spark.read.parquet(products_cleaned__file_path)
df.show(5)

In [ ]:
# Join main data with category names
df = df.join(categories, on='category_id', how='left')

In [ ]:
# convert booleans to 0,1

df = df.withColumn("has_listPrice", col("has_listPrice").cast("int"))
df = df.withColumn("isBestSeller", col("isBestSeller").cast("int"))

In [ ]:
# StringIndexing the category_name (convert strings to numeric indices)
indexer = StringIndexer(inputCol="category_name", outputCol="category_index")
df = indexer.fit(df).transform(df)

In [ ]:
# StringIndexing the category_name (convert strings to numeric indices)
indexer = StringIndexer(inputCol="category_name", outputCol="category_index")
df = indexer.fit(df).transform(df)
# Now df contains an additional 'category_encoded' column with one-hot encoded vectors

In [ ]:
# rating_weighted_reviews: Multiply stars by reviews to create a new feature
df = df.withColumn('rating_weighted_reviews', col('stars') * col('reviews'))

In [ ]:
# isPopular: Flag products with reviews above the 75th percentile
# Define a window to compute percentiles
windowSpec = Window.orderBy(col('reviews'))

# Compute the percentile rank of reviews
df = df.withColumn("percent_rank", percent_rank().over(windowSpec))

# Create 'isPopular' feature: 1 if above 75th percentile, 0 otherwise
df = df.withColumn("isPopular", (col("percent_rank") > 0.75).cast("int"))

In [ ]:
# price_log: Log-transform price
df = df.withColumn('price_log', log1p(col('price')))
# do we need to?

In [ ]:
# title_length
df = df.withColumn('title_length', length(col('title')))

In [ ]:
# Tokenize the titles
tokenizer = Tokenizer(inputCol="title", outputCol="title_tokens")
df = tokenizer.transform(df)

# Apply HashingTF to get term frequencies
hashingTF = HashingTF(inputCol="title_tokens", outputCol="raw_features", numFeatures=1000)
df = hashingTF.transform(df)

# Apply IDF to scale the term frequencies
idf = IDF(inputCol="raw_features", outputCol="title_tfidf")
idf_model = idf.fit(df)
df = idf_model.transform(df)

# Use the TF-IDF vectors in your classification model
assembler = VectorAssembler(inputCols=['title_tfidf', 'other_feature_1', 'other_feature_2'], outputCol='features')
df = assembler.transform(df)


In [ ]:
# Step 1: StringIndexing the category_name (convert strings to numeric indices)
indexer = StringIndexer(inputCol="category_name", outputCol="category_index")
df = indexer.fit(df).transform(df)

# Step 2: One-Hot Encoding the indexed category
encoder = OneHotEncoder(inputCols=["category_index"], outputCols=["category_encoded"])
df = encoder.fit(df).transform(df)

# Check the result
df.select("category_name", "category_index", "category_encoded").show(truncate=False)

In [ ]:
# # One hot encode the keywords

# from pyspark.sql.functions import array_contains

# # Loop through each keyword and create a binary column for its presence
# for i, keyword in enumerate(vocab):
#     df = df.withColumn(f'contains_{keyword}', array_contains(col("filtered_tokens"), keyword).cast("int"))

# # Show the updated dataframe with the new keyword columns
# df.select("title", *[f'contains_{keyword}' for keyword in vocab]).show(truncate=False)
